# Stage7 · 官方 subject LoRA:默认模式 vs KV-Cache 主体驱动对比

**问题**:挂官方 `subject_512` LoRA(按**默认模式**训练)时,开 `kv_cache=True` 掉多少质量?
这是 stage4 的**镜像方向**——stage4 里 feature_reuse LoRA 按独立条件模式训,吃亏的是 default;
这里 LoRA 按默认模式训,预期吃亏的是 kvcache。两组拼起来是 2×2 里"各自主场"的两个对角。

与 stage6(无 LoRA)的差异:

| | stage6 | stage7(本实验) |
|---|---|---|
| LoRA | 无(纯基座) | 官方 `Yuanshi/OminiControl` subject |
| Condition adapter | `None` | `"subject"`(仅条件分支走 LoRA,文本/主图分支仍走基座) |
| position_delta | `(0, size//16)` | 官方约定:512→`(0,32)`,1024→`(0,-32)` |
| image_guidance_scale | 1.0(默认) | **1.5**(dev 上 subject LoRA 必须 >1.0,两模式同加,保持公平) |
| 预期 | 基座忽略条件图(主体保持是 LoRA 能力) | 主体应被保持;看 kvcache 侧掉多少 |

前置:远程已运行 `python repro/download_subject_lora.py --which all`(LoRA 已进 HF cache,
offline 模式下可直接命中)。


In [ ]:
import os
os.chdir("..")  # 仓库根目录(与其它 repro notebook 一致)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import sys, json, time, random
sys.path.insert(0, "."); sys.path.insert(0, "repro")

import torch
# 与 stage3/probe 相同的兜底:torch 2.8 + diffusers 0.38 的 VAE conv_in 问题
torch.backends.cudnn.enabled = False

from PIL import Image
from kvcache_benchmark import load_pipeline, attach_lora
from omini.pipeline.flux_omini import Condition, generate

# ── 唯一需要改的开关:生成分辨率 ──────────────────────────────────────
TARGET_SIZE = 512                       # 512 / 1024(官方只发布了这两档 subject LoRA)
# ──────────────────────────────────────────────────────────────────
# LoRA / position_delta 随分辨率自动跟随官方约定(delta 符号任意,但必须与训练一致)
_LORA_BY_SIZE = {
    512:  ("omini/subject_512.safetensors",       (0, 32)),
    1024: ("omini/subject_1024_beta.safetensors", (0, -32)),
}
if TARGET_SIZE not in _LORA_BY_SIZE:
    raise ValueError(f"TARGET_SIZE={TARGET_SIZE} 无对应官方 subject LoRA,只能 512 或 1024")
LORA_WEIGHT, POS_DELTA = _LORA_BY_SIZE[TARGET_SIZE]

COND_SIZE   = TARGET_SIZE               # 主体驱动惯例:条件尺寸 = 目标尺寸
STEPS       = 28                        # 与 stage4-6 保持一致,便于横向比较
IMG_GS      = 1.5                       # dev + subject LoRA 唯一可调 CFG,官方推荐 1.5
SEEDS       = [42, 1234]
OUT_DIR     = f"repro/subject_lora_compare_{TARGET_SIZE}"  # 随 TARGET_SIZE 切换,多档并存
os.makedirs(OUT_DIR, exist_ok=True)
os.environ["OUT_DIR"] = OUT_DIR   # 同步给可视化 cell 的 shell magic 用
print(f"[DEBUG] setup: lora={LORA_WEIGHT} delta={POS_DELTA} steps={STEPS} "
      f"img_gs={IMG_GS} cond={COND_SIZE}px target={TARGET_SIZE}px out={OUT_DIR}")


In [ ]:
# 强制走本地 HF cache + 直接用 snapshot 路径(双保险,完全绕开网络)。
# subject LoRA 已由 download_subject_lora.py 提前入 cache,offline 下可命中。
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

LOCAL_FLUX_PATH = "/home/wuwenxuan03/.cache/huggingface/hub/models--black-forest-labs--FLUX.1-dev/snapshots/3de623fc3c33e44ffbe2bad470d0f45bccf2eb21"
print("LOCAL_FLUX_PATH exists:", os.path.isdir(LOCAL_FLUX_PATH))


In [ ]:
# 加载 pipeline + 官方 subject LoRA。
# attach_lora 内部做 set_adapters + device sweep(多卡 dispatch 下 PEFT 会把新 LoRA
# 放错卡,sweep 是必须的,见 kvcache_benchmark.py 失败 #11 注释)。
pipe = load_pipeline(LOCAL_FLUX_PATH, dispatch="auto", gpu0=2, gpu1=3)
attach_lora(pipe, "Yuanshi/OminiControl", LORA_WEIGHT, "subject")


In [ ]:
# 6 张主体条件图,与 stage6 完全同一组(便于两个实验逐行对照)。
# 前 5 条 prompt 逐字取自官方 examples/subject*.ipynb;book 自拟(官方无 subject 用例)。
CASES = [
    ("penguin", "assets/penguin.jpg",
     "On Christmas evening, on a crowded sidewalk, this item sits on the road, "
     "covered in snow and wearing a Christmas hat."),
    ("rc_car", "assets/rc_car.jpg",
     "A film style shot. On the moon, this item drives across the moon surface. "
     "The background is that Earth looms large in the foreground."),
    ("tshirt", "assets/tshirt.jpg",
     "On the beach, a lady sits under a beach umbrella. She's wearing this shirt and has "
     "a big smile on her face, with her surfboard hehind her. The sun is setting in the "
     "background. The sky is a beautiful shade of orange and purple."),
    ("clock", "assets/clock.jpg",
     "In a Bauhaus style room, this item is placed on a shiny glass table, with a vase of "
     "flowers next to it. In the afternoon sun, the shadows of the blinds are cast on the wall."),
    ("oranges", "assets/oranges.jpg",
     "A very close up view of this item. It is placed on a wooden table. The background is "
     "a dark room, the TV is on, and the screen is showing a cooking show."),
    ("book", "assets/book.jpg",  # prompt 自拟(官方无 subject 用例)
     "This item is placed on a wooden desk in a library, next to a cup of coffee, "
     "under the warm light of a reading lamp."),
]

def build_condition(image_path):
    # 官方预处理:center-crop 成正方形再 resize(pipeline 不会自动做)
    img = Image.open(image_path).convert("RGB")
    w, h = img.size; m = min(w, h)
    img = img.crop(((w - m) // 2, (h - m) // 2, (w + m) // 2, (h + m) // 2))
    img = img.resize((COND_SIZE, COND_SIZE))
    # 与 stage6 唯一的分支差异:adapter="subject" → 条件分支走官方 LoRA
    return img, Condition(img, "subject", position_delta=POS_DELTA)

def run_one(prompt, condition, seed, kv_cache):
    # WHY 每次重建 generator:两种模式必须从同一噪声出发,差异才全部归因于注意力模式
    g = torch.Generator(device="cuda:0").manual_seed(seed)
    t0 = time.perf_counter()
    out = generate(pipe, prompt=prompt, conditions=[condition],
                   height=TARGET_SIZE, width=TARGET_SIZE,
                   num_inference_steps=STEPS, guidance_scale=3.5,
                   image_guidance_scale=IMG_GS,   # dev + subject LoRA 必须 >1.0
                   generator=g, kv_cache=kv_cache)
    return out.images[0], time.perf_counter() - t0


In [ ]:
# 主循环:6 case × 2 seed × 2 模式 = 24 次生成
records = []
for name, path, prompt in CASES:
    cond_img, condition = build_condition(path)
    cond_img.save(f"{OUT_DIR}/{name}_cond.png")
    for seed in SEEDS:
        row = {"case": name, "seed": seed, "prompt": prompt}
        for mode, kv in (("default", False), ("kvcache", True)):
            img, dt = run_one(prompt, condition, seed, kv)
            fp = f"{OUT_DIR}/{name}_s{seed}_{mode}.png"
            img.save(fp)
            row[mode] = fp; row[f"{mode}_sec"] = round(dt, 1)
        records.append(row)
        print(f"[DEBUG] run_one: {name} seed={seed} "
              f"default={row['default_sec']}s kvcache={row['kvcache_sec']}s "
              f"speedup={row['default_sec']/row['kvcache_sec']:.2f}x")

with open(f"{OUT_DIR}/records.json", "w") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f"完成 {len(records)} 组对照,已写 {OUT_DIR}/records.json")


In [ ]:
# 可视化:复用独立脚本,产物 $OUT_DIR/comparison_grid.png
!python repro/visualize_quality_compare.py \
    --records $OUT_DIR/records.json \
    --out-dir $OUT_DIR


In [ ]:
# GSB 标注骨架(格式对齐 scripts/compute_gsb.py),盲评随机换边
rng = random.Random(0)
annotations, side_map = {}, {}
for r in records:
    cid = f"{r['case']}_s{r['seed']}"
    flip = rng.random() < 0.5
    side_map[cid] = {"A": "kvcache" if flip else "default",
                     "B": "default" if flip else "kvcache"}
    annotations[cid] = {"prompt": r["prompt"], "choices": {"quality": ""}}

with open(f"{OUT_DIR}/annotations.json", "w") as f:
    json.dump({"annotations": annotations}, f, ensure_ascii=False, indent=2)
with open(f"{OUT_DIR}/side_map.json", "w") as f:
    json.dump(side_map, f, ensure_ascii=False, indent=2)
print(f"标注后统计: python scripts/compute_gsb.py {OUT_DIR}/annotations.json")


## 读图指引

1. **主体保真度是第一观察点**:与 stage6 相反,这里两个模式都应该画出条件图里的主体。
   看 default 与 kvcache 谁的主体更像条件图(纹理、配色、结构)。
2. **预期方向**:`subject_512` 按默认模式训练,条件分支在训练时每步都看得到演化中的主图
   latent;开 `kv_cache=True` 后条件 KV 冻结在第 0 步,对这块 LoRA 是分布外推理——
   **预期 kvcache 侧主体保真度/细节下降**。掉多少、是否可接受,正是本实验要量化的。
3. **速度**:条件 token 恒占序列一半,KV-Cache 收益上限恒定;但 `image_guidance_scale=1.5`
   会让每步多跑一次空条件 CFG 分支,绝对时间比 stage6 长,speedup 比例仍可对照。
4. 与 stage4(feature_reuse LoRA,吃亏方 = default)拼起来即 2×2 的两个"客场"对角;
   剩余空格需要补训 `independent_condition: false` 的对照 LoRA 才能填上。

标注 `annotations.json` 的 `choices.quality`(G = default 好,B = kvcache 好,S = 持平,
按 `side_map.json` 还原盲评换边)后运行 `python scripts/compute_gsb.py` 统计。
